In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

Metrics: consistency and stability

In [2]:
from src.data.shapenet import ShapeNetConsistencyTest, ShapeNetStabilityTest


label_in_use = "0"
num_points = 1024
resample = False
shapenet_path = "data/shapenet"
model_k = 20

cns_dset = ShapeNetConsistencyTest(
    label_in_use, num_points, resample, shapenet_path
)
stb_dset = ShapeNetStabilityTest(
    label_in_use, num_points, resample, model_k, shapenet_path
)
cns_dloader = DataLoader(
    cns_dset,
    batch_size=128,
    num_workers=4,
    shuffle=False,
    drop_last=False
)
stb_dloader = DataLoader(
    stb_dset,
    batch_size=1,
    num_workers=4,
    shuffle=False,
    drop_last=False
)

In [3]:
from src.rotation import cal_angular_std_from_rotations
from src.task.orientation import mean


@torch.inference_mode()
def eval_consistency(model, data_loader, device):
    model.eval()

    outputs = dict()
    for batch in tqdm(data_loader):
        pcd = batch["pcd"]
        label = batch["label"]

        rots = model(pcd.to(device), "cns")
        rots = rots.cpu()

        if label in outputs.keys():
            outputs[label].append(rots)
        else:
            outputs[label] = [rots]

    classwise_std = dict()
    for label, rots in outputs.items():
        rots = torch.vstack(rots)
        std = cal_angular_std_from_rotations(rots).item()
        classwise_std[label] = std

    mean_std = mean(list(classwise_std.values()))

    return mean_std


@torch.inference_mode()
def eval_stability(model, data_loader, device): # batch size should be 1.
    model.eval()

    stds = []
    for batch in tqdm(data_loader):
        pcd = batch["pcd"].squeeze(0).to(device)
        knn_idx = batch["knn_indices"].squeeze(0).to(device)
        rots_aug = batch["rots"].squeeze(0).to(device)

        rots = model(pcd, "stb", rots_aug)
        rots_can = torch.einsum("bij, bjk -> bik", rots, rots_aug)
        std = cal_angular_std_from_rotations(rots_can).item()
        stds.append(std)

    mean_std = mean(stds)

    return mean_std

In [4]:
class PerferctModel(nn.Module):
    def forward(self, x, mode, rots=None):
        bsz = len(x)

        if mode == "cns":
            return self.forward_cns(bsz)
        else:
            return self.forward_stb(rots)

    def forward_cns(self, bsz):
        return torch.eye(3).unsqueeze(0).repeat(bsz, 1, 1)

    def forward_stb(self, rots):
        return rots.transpose(1, 2)

In [5]:
device = torch.device('cpu')
model = PerferctModel()

cns = eval_consistency(model, cns_dloader, device)
print(cns)
stb = eval_stability(model, stb_dloader, device)
print(stb)

100%|██████████| 7/7 [00:01<00:00,  6.11it/s]


0.0


100%|██████████| 809/809 [00:21<00:00, 37.02it/s]

0.007161758870015333
